# RNA implementation to SCICoNE

In [1]:
#import libraries

import scicone
import numpy as np
import pickle
import subprocess
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy import io
import scanpy as sc
import anndata
import pyranges
import gseapy as gp
import utils2

# Set up SCICoNE
install_path = '/cluster/work/bewi/members/andress/pylabs/SCICoNE/build/'
install_path_local = "/home/andress/pylabs/SCICoNE_lab/build/"
temporary_outpath = './'

seed = 42 # for reproducibility

np.random.seed(seed)

# Create SCICoNE object
sci = scicone.SCICoNE(install_path_local, temporary_outpath, verbose=False)

Using binaries at /home/andress/pylabs/SCICoNE_lab/build/


In [10]:


# Define paths
scdna_path_ssh = '/cluster/work/bewi/members/andress/SCICoNE_lab/rna_imp/clonealign-processed-data/SA501/cnv/'
scrna_path_ssh = '/cluster/work/bewi/members/andress/SCICoNE_lab/rna_imp/clonealign-processed-data/SA501/10X/20171026_SA501X2XB00096/outs/filtered_gene_bc_matrices/hg19/'

scdna_path_local = '/home/andress/pylabs/SCICoNE_lab/rna_imp/clonealign-processed-data/SA501/cnv'
scrna_path_local = '/home/andress/pylabs/SCICoNE_lab/rna_imp/clonealign-processed-data/SA501/10X/20171026_SA501X2XB00096/outs/filtered_gene_bc_matrices/hg19'

# Try SSH paths first
try:
    if os.path.exists(scdna_path_ssh) and os.path.exists(scrna_path_ssh):
        scdna_path = scdna_path_ssh
        scrna_path = scrna_path_ssh
    else:
        raise FileNotFoundError("SSH paths are not accessible.")
except FileNotFoundError:
    # Use local paths
    if os.path.exists(scdna_path_local) and os.path.exists(scrna_path_local):
        scdna_path = scdna_path_local
        scrna_path = scrna_path_local
    else:
        raise FileNotFoundError("Neither SSH nor local paths are accessible.")

FileNotFoundError: Neither SSH nor local paths are accessible.

In [27]:
annot = sc.queries.biomart_annotations(
        "hsapiens",
        ["ensembl_gene_id", "start_position", "end_position", "chromosome_name"],
    ).set_index("ensembl_gene_id")

annot = annot.reset_index()
annot = annot.rename(columns={'start_position':'Start', 'end_position': 'End', 'chromosome_name': 'Chromosome'})

gr_annotations = pyranges.from_dict(annot.to_dict())

gr_annotations.head()

,ensembl_gene_id,Start,End,Chromosome
0,ENSG00000142611,3069168,3438621,1
1,ENSG00000284616,5301928,5307394,1
2,ENSG00000157911,2403964,2413797,1
3,ENSG00000260972,5492978,5494674,1
4,ENSG00000224340,10054445,10054781,1
5,ENSG00000229280,4175528,4175899,1
6,ENSG00000142655,10472288,10630758,1
7,ENSG00000232596,4571481,4594016,1


In [28]:
gr_cnvs = pyranges.from_dict(cnvs.rename(columns={'start':'Start', 'end': 'End', 'chr': 'Chromosome'}).to_dict())


gr_annotated_cnvs = gr_cnvs.join(gr_annotations).drop(like="_b").cluster().df\
            .drop_duplicates('ensembl_gene_id')\
            .reset_index(drop=True).drop(columns='Cluster')

n_genes = gr_annotated_cnvs.shape[0]

#print the shape of the annotated CNVs
print(gr_annotated_cnvs.shape)

n_genes


NameError: name 'cnvs' is not defined

In [29]:
chr_var_names = dict()
for chromosome in gr_annotated_cnvs['Chromosome'].unique():
    chr_var_names[chromosome] = gr_annotated_cnvs.query(f' Chromosome=="{chromosome}" ')['ensembl_gene_id'].values

print(chr_var_names)

NameError: name 'gr_annotated_cnvs' is not defined

# Read scRNA data

In [30]:
mat = io.mmread(f'{scrna_path}/matrix.mtx').toarray()
barcodes = pd.read_csv(f'{scrna_path}/barcodes.tsv', sep='\t', header=None)
genes = pd.read_csv(f'{scrna_path}/genes.tsv', sep='\t', header=None)

#Matrix gene by cells required by SCICoNE
gene_by_cells = pd.DataFrame(mat, index=genes[0].values, columns=barcodes[0].values)

cells_by_genes = gene_by_cells.T
print(cells_by_genes.shape[0])
print(cells_by_genes.head())

2470
                    ENSG00000243485  ENSG00000237613  ENSG00000186092  \
AAACCTGAGATCCGAG-1                0                0                0   
AAACCTGAGATGCCAG-1                0                0                0   
AAACCTGTCGACAGCC-1                0                0                0   
AAACCTGTCTTCAACT-1                0                0                0   
AAACGGGAGCAATCTC-1                0                0                0   

                    ENSG00000238009  ENSG00000239945  ENSG00000237683  \
AAACCTGAGATCCGAG-1                0                0                0   
AAACCTGAGATGCCAG-1                0                0                0   
AAACCTGTCGACAGCC-1                0                0                0   
AAACCTGTCTTCAACT-1                0                0                0   
AAACGGGAGCAATCTC-1                0                0                0   

                    ENSG00000239906  ENSG00000241599  ENSG00000228463  \
AAACCTGAGATCCGAG-1                0         

In [47]:
adata = anndata.AnnData(pd.DataFrame(mat.T, index=barcodes[0].values, columns=genes[0].values))
adata.var_names_make_unique()
adata.var['ensembl_gene_id'] = genes[0].values
adata.var['gene_id'] = genes[1].values
print(adata.obs_names[:5])  # First 5 observation names (e.g., barcodes)
print(adata.var_names[:5])  # First 5 variable names (e.g., gene names)
print(adata.shape)  # Should return (number of cells, number of genes)

Index(['AAACCTGAGATCCGAG-1', 'AAACCTGAGATGCCAG-1', 'AAACCTGTCGACAGCC-1',
       'AAACCTGTCTTCAACT-1', 'AAACGGGAGCAATCTC-1'],
      dtype='object')
Index(['ENSG00000243485', 'ENSG00000237613', 'ENSG00000186092',
       'ENSG00000238009', 'ENSG00000239945'],
      dtype='object')
(2470, 32738)


# Pre-processing 

In [48]:
#basic filtering

sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

#filter 

adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
adata = adata[adata.obs.n_genes_by_counts < 2500, :] #doublets 
adata = adata[adata.obs.pct_counts_mt < 5, :] #mitochondrial quality control

In [49]:
adata.raw = adata #checkpoint 

In [ ]:
#further qc filtering

#normalize the data
sc.pp.normalize_total(adata, target_sum=1e4)

#filter highly variable genes
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
adata = adata[:, adata.var.highly_variable]
sc.pp.regress_out(adata, ['total_counts', 'pct_counts_mt'])

sc.pp.scale(adata, max_value=10)
#smoothing average

#cluster data

sc.tl.pca(adata, svd_solver='arpack')
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)
sc.tl.leiden(adata)
sc.pl.umap(adata, color='leiden')

/home/andress/miniconda3/envs/scicone_env/lib/python3.11/site-packages/scanpy/preprocessing/_simple.py:709: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)


KeyError: "Could not find 'umap' or 'X_umap' in .obsm"

In [56]:
#save the data
adata.write_h5ad(f'{temporary_outpath}/adata.h5wad')

In [51]:
# Get gene coordinates
df_annotations = gr_annotations.df.set_index('ensembl_gene_id')

df_exp_annotations = df_annotations.loc[df_annotations.index.intersection(adata.var['ensembl_gene_id'])]\
                        .reset_index().rename(columns={'index':'ensembl_gene_id'})

adata.var_names = adata.var['ensembl_gene_id'].values

adata = adata[:,df_exp_annotations['ensembl_gene_id']]

In [52]:
df_exp_annotations_sorted = df_exp_annotations.sort_values(by=['Chromosome', 'End'])


df_exp_annotations_sorted.head()

,ensembl_gene_id,Start,End,Chromosome
191,ENSG00000228327,764723,774280,1
248,ENSG00000237491,778739,810066,1
84,ENSG00000272141,1169357,1170343,1
170,ENSG00000078808,1216931,1232031,1
129,ENSG00000221978,1385711,1399335,1


In [53]:
# Create a clean copy with corrected chromosome names
df_exp_annotations_2 = df_exp_annotations_sorted.copy()
df_exp_annotations_2['Chromosome'] = df_exp_annotations_2['Chromosome'].astype(str).str.strip()

standard_chromosomes = [str(i) for i in range(1, 23)] + ["X", "Y"]
filtered_df = df_exp_annotations_2[df_exp_annotations_2['Chromosome'].isin(standard_chromosomes)]

idx_max = filtered_df.groupby('Chromosome')['End'].idxmax()

# Sort by chromosome order first, then by End
chromosome_stops_df = filtered_df.loc[idx_max, ['Chromosome', 'End', 'ensembl_gene_id']].reset_index(drop=True)

# Create custom chromosome ordering
chromosome_order = {str(i): i for i in range(1, 23)}
chromosome_order.update({'X': 23, 'Y': 24})
chromosome_stops_df['chr_order'] = chromosome_stops_df['Chromosome'].map(chromosome_order)

# Sort by chromosome order
chromosome_stops_df = chromosome_stops_df.sort_values(by='chr_order').drop('chr_order', axis=1).reset_index(drop=True)

print(chromosome_stops_df)

   Chromosome        End  ensembl_gene_id
0           1  248859144  ENSG00000171163
1           2  241686944  ENSG00000168393
2           3  197960142  ENSG00000114473
3           4  187415758  ENSG00000250620
4           5  180591499  ENSG00000161055
5           6  169725566  ENSG00000130024
6           7  158830253  ENSG00000117868
7           8  144502121  ENSG00000160972
8           9  137618906  ENSG00000203993
9          10  128126423  ENSG00000148773
10         11  126440344  ENSG00000110080
11         12  133063304  ENSG00000198040
12         13  108308484  ENSG00000102524
13         14  105470729  ENSG00000182979
14         15  101277500  ENSG00000131871
15         16   90106316  ENSG00000260507
16         17   82840022  ENSG00000141579
17         18   80179839  ENSG00000267270
18         19   58605223  ENSG00000267858
19         20   63980008  ENSG00000130590
20         21   46605208  ENSG00000160307
21         22   50548994  ENSG00000272666
22          X  154547572  ENSG0000

In [54]:
chromosome_stops = []

for gene_id in chromosome_stops_df['ensembl_gene_id']:
    index = np.where(adata.var_names == gene_id)[0][0]
    chromosome_stops.append(int(index))
    
print(chromosome_stops)
print(len(chromosome_stops))

[48, 271, 468, 620, 699, 753, 911, 993, 1103, 1172, 1220, 1361, 1428, 1482, 1604, 1633, 1728, 1826, 1939, 1973, 2007, 2056, 2095]
23


In [70]:
import numpy as np
import scanpy as sc
import logging

def smooth_expression(adata, var_names=None, window_size=10, clip=3):
    """
    Apply expression smoothing to an AnnData object
    
    Parameters:
    -----------
    adata : AnnData
        AnnData object containing gene expression data
    var_names : list, dict or None
        Gene names to include in smoothing. If dict, keys are chromosome names and values are gene lists
    window_size : int
        Size of the sliding window for smoothing
    clip : float
        Threshold for clipping extreme values
        
    Returns:
    --------
    None, adds 'scaled' and 'smoothed' layers to the adata object
    """
    logger = logging.getLogger(__name__)
    
    # Scale the data first
    mat = sc.pp.scale(adata.X, copy=True)
    adata.layers["scaled"] = mat
    
    # For stand-alone smoothing (no chromosome regions)
    if var_names is None or not isinstance(var_names, dict):
        # Apply clipping
        clipped_mat = np.clip(mat, -np.abs(clip), np.abs(clip))
        
        # Apply smoothing
        half_window = int(window_size / 2)
        smoothed = np.zeros(clipped_mat.shape)
        
        for ii in range(clipped_mat.shape[1]):
            left = max(0, ii - half_window)
            right = min(ii + half_window, clipped_mat.shape[1] - 1)
            if left != right:
                smoothed[:, ii] = np.mean(clipped_mat[:, left:right], axis=1)
        
        adata.layers["smoothed"] = smoothed
    
    # For chromosome-specific smoothing
    else:
        smoothed_mat = []
        
        for region in var_names:
            region_slice = adata[:, var_names[region]].layers["scaled"]
            
            # Apply clipping
            clipped = np.clip(region_slice, -np.abs(clip), np.abs(clip))
            
            # Apply smoothing
            half_window = int(window_size / 2)
            smoothed = np.zeros(clipped.shape)
            
            for ii in range(clipped.shape[1]):
                left = max(0, ii - half_window)
                right = min(ii + half_window, clipped.shape[1] - 1)
                if left != right:
                    smoothed[:, ii] = np.mean(clipped[:, left:right], axis=1)
            
            smoothed_mat.append(smoothed)
        
        adata.layers["smoothed"] = np.concatenate(smoothed_mat, axis=1)
    
    logger.info(f'Smoothed gene expression is stored in `adata.layers["smoothed"]`')

In [73]:
#Smoothed average

smooth_expression(adata, var_names=None, window_size=50, clip=3)

adata.shape()

TypeError: 'tuple' object is not callable

In [74]:
# Check if 'smoothed' layer exists
print("Layers in adata:", list(adata.layers.keys()))

# Check shape of the smoothed layer
if 'smoothed' in adata.layers:
    print("Shape of smoothed layer:", adata.layers['smoothed'].shape)
    
    # Look at the first few values of original and smoothed data
    print("\nFirst 5 values of first cell (original):")
    print(adata.X[0, :5])
    
    print("\nFirst 5 values of first cell (scaled):")
    print(adata.layers['scaled'][0, :5])
    
    print("\nFirst 5 values of first cell (smoothed):")
    print(adata.layers['smoothed'][0, :5])
    
    # Calculate difference between original and smoothed
    print("\nMean absolute difference between scaled and smoothed:")
    print(np.mean(np.abs(adata.layers['scaled'] - adata.layers['smoothed'])))

# adata shape (should be called without parentheses)
print("\nAnnData object shape:", adata.shape)

Layers in adata: ['scaled', 'smoothed']
Shape of smoothed layer: (1537, 2116)

First 5 values of first cell (original):
[-0.08576872 -0.20426699 -0.34306103 -0.34698462 -0.23200266]

First 5 values of first cell (scaled):
[-0.09069284 -0.20426699 -0.34306103 -0.34698462 -0.23200266]

First 5 values of first cell (smoothed):
[0.03876309 0.03352828 0.02706704 0.01539836 0.0104172 ]

Mean absolute difference between scaled and smoothed:
0.41875800505906524

AnnData object shape: (1537, 2116)


In [ ]:
#save adata ojects

def save_adata_objects(adata, output_dir=temporary_outpath):
    """
    Save copies of adata.raw and smoothed average adata locally.
    
    Parameters:
        adata (AnnData): The AnnData object containing single-cell RNA data.
        output_dir (str): Directory to save the files.
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # Save adata.raw if it exists
    if adata.raw is not None:
        adata_raw_copy = adata.raw.to_adata()
        adata_raw_copy.write(os.path.join(output_dir, "adata_raw.h5ad"))
        print("Saved adata.raw as adata_raw.h5ad")
    
    # Save the smoothed average adata
    adata.write(os.path.join(output_dir, "adata_smoothed.h5ad"))
    print("Saved smoothed adata as adata_smoothed.h5ad")


In [ ]:
save_adata_objects(adata)
save_adata_objects(adata.raw)

In [ ]:
def cluster_cells(adata, resolution=1.0):
    """
    Cluster cells in a copy of the AnnData object using the Leiden algorithm.
    
    Parameters:
        adata (AnnData): The AnnData object containing single-cell RNA data.
        resolution (float): Resolution parameter for the Leiden clustering.
    
    Returns:
        AnnData: A new AnnData object (adata_cluster) with clustering results added.
    """
    # Create a copy of the adata object
    adata_cluster = adata.copy()
    
    # Perform PCA
    sc.tl.pca(adata_cluster)
    
    # Compute the neighborhood graph
    sc.pp.neighbors(adata_cluster, n_neighbors=15, use_rep='X_pca')
    
    # Perform clustering using the Leiden algorithm
    sc.tl.leiden(adata_cluster, resolution=resolution)
    
    # Add clustering results to adata_cluster.obs
    print("Clustering completed. Results stored in adata_cluster.obs['leiden']")
    return adata_cluster

# Example usage
# Assuming `adata` is your AnnData object
save_adata_objects(adata)
adata_cluster = cluster_cells(adata)

# Run SCICoNE

In [ ]:
#original data
sci.detect_breakpoints(adata, window_size=100, threshold=3, input_breakpoints=chromosome_stops)
sci.learn_tree(ful=False)
sci.plot_tree()

KeyboardInterrupt: 

In [ ]:
#smoothed average

#clustered data 

sci.detect_breakpoints(adata_cluster, window_size=100, threshold=3, input_breakpoints=chromosome_stops)
sci.learn_tree(ful=False)
sci.plot_tree()